In [1]:
# ============================================================
# NOTEBOOK 5: FULL PIPELINE DEMO
# ============================================================
# This notebook demonstrates the complete end-to-end pipeline
# ============================================================

import sys
import os
import warnings
warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath(".."))
from config import *

import pandas as pd
import numpy as np
import joblib

print("✅ Setup complete")

✅ Setup complete


In [2]:
# ============================================================
# LOAD ALL COMPONENTS
# ============================================================

# Load scaler
scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
print("✅ Scaler loaded")

# Load feature list
FINAL_FEATURES = joblib.load(os.path.join(MODEL_DIR, "feature_list.pkl"))
print(f"✅ Feature list loaded ({len(FINAL_FEATURES)} features)")

# Load all models
models = {}
for disease in DISEASE_COLUMNS:
    models[disease] = joblib.load(os.path.join(MODEL_DIR, f"{disease}_model.pkl"))
print(f"✅ All {len(models)} models loaded")

# Load performance results
import json
with open(os.path.join(REPORT_DIR, "model_results.json"), "r") as f:
    performance = json.load(f)
print("✅ Performance metrics loaded")

✅ Scaler loaded
✅ Feature list loaded (36 features)
✅ All 6 models loaded
✅ Performance metrics loaded


In [3]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def calculate_bmi(weight_kg, height_cm):
    height_m = height_cm / 100
    return round(weight_kg / (height_m ** 2), 1)

def get_bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"

def engineer_features(patient_dict):
    p = patient_dict.copy()

    p["Calories_Per_kg_BMI"] = p["Daily_Calories_kcal"] / (p["BMI"] + 1)
    p["Carb_Calorie_Ratio"] = (p["Carbohydrates_g"] * 4) / (p["Daily_Calories_kcal"] + 1)
    p["Protein_Calorie_Ratio"] = (p["Protein_g"] * 4) / (p["Daily_Calories_kcal"] + 1)
    p["Fat_Calorie_Ratio"] = (p["Total_Fat_g"] * 9) / (p["Daily_Calories_kcal"] + 1)
    p["Sugar_to_Fiber_Ratio"] = (p["Total_Sugar_g"] + 1) / (p["Fiber_g"] + 1)
    p["Added_Sugar_Pct"] = (p["Added_Sugar_g"] + 1) / (p["Total_Sugar_g"] + 1)
    p["Unhealthy_Fat_Ratio"] = (p["Saturated_Fat_g"] + p["Trans_Fat_g"]) / (p["Total_Fat_g"] + 1)
    p["Healthy_Fat_g"] = max(0, p["Total_Fat_g"] - p["Saturated_Fat_g"] - p["Trans_Fat_g"])
    p["Sodium_Potassium_Ratio"] = (p["Sodium_mg"] + 1) / (p["Potassium_mg"] + 1)
    p["Calcium_Iron_Sum"] = p["Calcium_mg"] + p["Iron_mg"]
    p["Vitamin_Score"] = p["Vitamin_D_IU"] / 100 + p["Vitamin_B12_mcg"]
    p["Activity_Water_Score"] = p["Physical_Activity_min"] * p["Water_Intake_L"]

    act = p["Physical_Activity_min"]
    p["Activity_Level"] = 0 if act <= 15 else 1 if act <= 30 else 2 if act <= 60 else 3 if act <= 150 else 4

    bmi = p["BMI"]
    p["BMI_Category"] = 0 if bmi < 18.5 else 1 if bmi < 25 else 2 if bmi < 30 else 3

    age = p["Age"]
    p["Age_Group"] = 0 if age <= 25 else 1 if age <= 35 else 2 if age <= 45 else 3 if age <= 55 else 4 if age <= 65 else 5

    p["Health_Score"] = min(100, max(0, (
        (p["Fiber_g"] / 38) * 20 +
        (1 - p["Added_Sugar_g"] / 100) * 20 +
        (1 - p["Sodium_mg"] / 5000) * 20 +
        (p["Physical_Activity_min"] / 150) * 20 +
        (p["Water_Intake_L"] / 3.7) * 20
    )))

    return p

def predict_risks(patient_data):
    if "Weight_kg" in patient_data and "Height_cm" in patient_data:
        patient_data["BMI"] = calculate_bmi(patient_data["Weight_kg"], patient_data["Height_cm"])

    patient_engineered = engineer_features(patient_data)
    patient_df = pd.DataFrame([{k: patient_engineered[k] for k in FINAL_FEATURES}])
    patient_scaled = scaler.transform(patient_df)

    predictions = {}
    for disease in DISEASE_COLUMNS:
        model = models[disease]
        pred = model.predict(patient_scaled)[0]
        prob = model.predict_proba(patient_scaled)[0][1]
        predictions[disease] = {
            "prediction": int(pred),
            "risk_level": "High Risk" if pred == 1 else "Low Risk",
            "probability": round(float(prob) * 100, 1),
            "confidence": round(float(max(prob, 1 - prob)) * 100, 1),
        }
    return predictions

print("✅ All utility functions loaded")

✅ All utility functions loaded


In [4]:
# ============================================================
# MODEL PERFORMANCE SUMMARY
# ============================================================

print("=" * 70)
print("📊 MODEL PERFORMANCE SUMMARY")
print("=" * 70)

for disease in DISEASE_COLUMNS:
    p = performance[disease]
    print(f"\n🔹 {disease.replace('_', ' ')}")
    print(f"   Accuracy: {p['accuracy']:.4f}  |  F1: {p['f1_score']:.4f}  |  AUC: {p['auc_roc']:.4f}")

📊 MODEL PERFORMANCE SUMMARY

🔹 Diabetes Risk
   Accuracy: 1.0000  |  F1: 1.0000  |  AUC: 1.0000

🔹 Hypertension Risk
   Accuracy: 0.9988  |  F1: 0.9973  |  AUC: 1.0000

🔹 Heart Disease Risk
   Accuracy: 1.0000  |  F1: 1.0000  |  AUC: 1.0000

🔹 Obesity Risk
   Accuracy: 1.0000  |  F1: 1.0000  |  AUC: 1.0000

🔹 Anemia Risk
   Accuracy: 1.0000  |  F1: 1.0000  |  AUC: 1.0000

🔹 Kidney Disease Risk
   Accuracy: 0.9992  |  F1: 0.9964  |  AUC: 1.0000


In [5]:
# ============================================================
# DEMO - MULTIPLE PATIENT SCENARIOS
# ============================================================

patients = {
    "Unhealthy Patient (High Risk)": {
        "Age": 55, "Gender": 0, "Weight_kg": 95, "Height_cm": 170,
        "Daily_Calories_kcal": 3200, "Carbohydrates_g": 400,
        "Protein_g": 60, "Total_Fat_g": 150, "Saturated_Fat_g": 55,
        "Trans_Fat_g": 5, "Total_Sugar_g": 150, "Added_Sugar_g": 100,
        "Fiber_g": 10, "Sodium_mg": 4500, "Potassium_mg": 1500,
        "Calcium_mg": 400, "Iron_mg": 5, "Vitamin_D_IU": 200,
        "Vitamin_B12_mcg": 1.0, "Physical_Activity_min": 10,
        "Water_Intake_L": 1.0,
    },
    "Healthy Patient (Low Risk)": {
        "Age": 28, "Gender": 1, "Weight_kg": 60, "Height_cm": 165,
        "Daily_Calories_kcal": 1900, "Carbohydrates_g": 240,
        "Protein_g": 70, "Total_Fat_g": 60, "Saturated_Fat_g": 12,
        "Trans_Fat_g": 0, "Total_Sugar_g": 30, "Added_Sugar_g": 10,
        "Fiber_g": 32, "Sodium_mg": 1500, "Potassium_mg": 3800,
        "Calcium_mg": 1100, "Iron_mg": 14, "Vitamin_D_IU": 900,
        "Vitamin_B12_mcg": 4.0, "Physical_Activity_min": 60,
        "Water_Intake_L": 2.8,
    },
    "Moderate Risk Patient": {
        "Age": 40, "Gender": 0, "Weight_kg": 82, "Height_cm": 175,
        "Daily_Calories_kcal": 2400, "Carbohydrates_g": 300,
        "Protein_g": 85, "Total_Fat_g": 90, "Saturated_Fat_g": 25,
        "Trans_Fat_g": 1, "Total_Sugar_g": 70, "Added_Sugar_g": 40,
        "Fiber_g": 20, "Sodium_mg": 2800, "Potassium_mg": 2500,
        "Calcium_mg": 800, "Iron_mg": 10, "Vitamin_D_IU": 500,
        "Vitamin_B12_mcg": 2.5, "Physical_Activity_min": 25,
        "Water_Intake_L": 1.8,
    },
}

for name, patient in patients.items():
    print("\n" + "=" * 70)
    print(f"👤 PATIENT: {name}")
    print("=" * 70)

    preds = predict_risks(patient)
    bmi = patient.get("BMI", calculate_bmi(patient["Weight_kg"], patient["Height_cm"]))

    print(f"   BMI: {bmi} ({get_bmi_category(bmi)})")
    print(f"\n   Disease Predictions:")

    for disease, pred in preds.items():
        icon = "🔴" if pred["prediction"] == 1 else "🟢"
        print(f"   {icon} {disease.replace('_', ' ')}: "
              f"{pred['risk_level']} ({pred['probability']}%)")


👤 PATIENT: Unhealthy Patient (High Risk)
   BMI: 32.9 (Obese)

   Disease Predictions:
   🔴 Diabetes Risk: High Risk (99.9%)
   🔴 Hypertension Risk: High Risk (100.0%)
   🔴 Heart Disease Risk: High Risk (99.8%)
   🔴 Obesity Risk: High Risk (99.9%)
   🔴 Anemia Risk: High Risk (99.7%)
   🔴 Kidney Disease Risk: High Risk (99.9%)

👤 PATIENT: Healthy Patient (Low Risk)
   BMI: 22.0 (Normal)

   Disease Predictions:
   🟢 Diabetes Risk: Low Risk (0.0%)
   🟢 Hypertension Risk: Low Risk (0.0%)
   🟢 Heart Disease Risk: Low Risk (0.0%)
   🟢 Obesity Risk: Low Risk (0.0%)
   🟢 Anemia Risk: Low Risk (0.0%)
   🟢 Kidney Disease Risk: Low Risk (0.0%)

👤 PATIENT: Moderate Risk Patient
   BMI: 26.8 (Overweight)

   Disease Predictions:
   🟢 Diabetes Risk: Low Risk (0.0%)
   🔴 Hypertension Risk: High Risk (50.0%)
   🟢 Heart Disease Risk: Low Risk (0.5%)
   🟢 Obesity Risk: Low Risk (0.0%)
   🟢 Anemia Risk: Low Risk (0.0%)
   🟢 Kidney Disease Risk: Low Risk (0.0%)


In [6]:
# ============================================================
# QUICK PREDICT FUNCTION (For Easy Use)
# ============================================================

def quick_predict(age, gender, weight_kg, height_cm,
                  calories, carbs, protein, fat, sat_fat, trans_fat,
                  sugar, added_sugar, fiber, sodium, potassium,
                  calcium, iron, vit_d, vit_b12,
                  activity_min, water_l):
    """
    Quick prediction with positional arguments.
    Gender: 0=Male, 1=Female
    """

    patient = {
        "Age": age, "Gender": gender,
        "Weight_kg": weight_kg, "Height_cm": height_cm,
        "Daily_Calories_kcal": calories,
        "Carbohydrates_g": carbs, "Protein_g": protein,
        "Total_Fat_g": fat, "Saturated_Fat_g": sat_fat,
        "Trans_Fat_g": trans_fat, "Total_Sugar_g": sugar,
        "Added_Sugar_g": added_sugar, "Fiber_g": fiber,
        "Sodium_mg": sodium, "Potassium_mg": potassium,
        "Calcium_mg": calcium, "Iron_mg": iron,
        "Vitamin_D_IU": vit_d, "Vitamin_B12_mcg": vit_b12,
        "Physical_Activity_min": activity_min,
        "Water_Intake_L": water_l,
    }

    preds = predict_risks(patient)

    print(f"\n🏥 Results for {age}yr {'Female' if gender==1 else 'Male'}, "
          f"BMI: {calculate_bmi(weight_kg, height_cm)}")
    print("─" * 40)

    for disease, pred in preds.items():
        icon = "🔴" if pred["prediction"] == 1 else "🟢"
        print(f"  {icon} {disease.replace('_', ' ')}: {pred['probability']}%")

    return preds

# Example usage
print("💡 Usage example:")
print('quick_predict(45, 1, 70, 154, 2800, 350, 90, 130, 45, 2, 120, 90, 15, 3500, 2000, 500, 7, 2190, 1.8, 20, 1.2)')

💡 Usage example:
quick_predict(45, 1, 70, 154, 2800, 350, 90, 130, 45, 2, 120, 90, 15, 3500, 2000, 500, 7, 2190, 1.8, 20, 1.2)


In [7]:
# ============================================================
# FINAL TEST
# ============================================================

result = quick_predict(
    age=45, gender=1,
    weight_kg=70, height_cm=154,
    calories=2800, carbs=350, protein=90,
    fat=130, sat_fat=45, trans_fat=2,
    sugar=120, added_sugar=90, fiber=15,
    sodium=3500, potassium=2000,
    calcium=500, iron=7,
    vit_d=2190, vit_b12=1.8,
    activity_min=20, water_l=1.2
)


🏥 Results for 45yr Female, BMI: 29.5
────────────────────────────────────────
  🟢 Diabetes Risk: 0.2%
  🔴 Hypertension Risk: 100.0%
  🔴 Heart Disease Risk: 99.8%
  🟢 Obesity Risk: 0.0%
  🔴 Anemia Risk: 98.5%
  🔴 Kidney Disease Risk: 99.9%
